In [10]:
import pandas as pd
import numpy as np

def compute_did(df, group_col='group', period_col='period', outcome_col='outcome'):
    # ── Input validation ──────────────────────────────────
    assert group_col   in df.columns, f"{group_col} not in df"
    assert period_col  in df.columns, f"{period_col} not in df"
    assert outcome_col in df.columns, f"{outcome_col} not in df"

    # ── Step 1: Compute group × period means ──────────────
    means = df.groupby([group_col, period_col])[outcome_col].mean()

    # ── Step 2: Extract four numbers ──────────────────────
    treat_pre    = means.loc[('treat',   'pre')]
    treat_post   = means.loc[('treat',   'post')]
    control_pre  = means.loc[('control', 'pre')]
    control_post = means.loc[('control', 'post')]

    # ── Step 3: DiD formula ───────────────────────────────
    delta_treat   = treat_post   - treat_pre
    delta_control = control_post - control_pre
    did_estimate  = delta_treat  - delta_control

    # ── Step 4: Conclusion ────────────────────────────────
    if did_estimate > 0:
        conclusion = 'Positive effect'
    elif did_estimate < 0:
        conclusion = 'Negative effect'
    else:
        conclusion = 'No effect detected'

    return {
        'treat_pre'    : treat_pre,
        'treat_post'   : treat_post,
        'control_pre'  : control_pre,
        'control_post' : control_post,
        'delta_treat'  : delta_treat,
        'delta_control': delta_control,
        'did_estimate' : did_estimate,
        'conclusion'   : conclusion
    }







In [12]:
# ── Quick test ────────────────────────────────────────────
data = {
    'group'  : ['treat','treat','treat','treat','control','control','control','control'],
    'period' : ['pre','pre','post','post','pre','pre','post','post'],
    'outcome': [10, 12, 20, 22, 10, 12, 13, 15]
}
df = pd.DataFrame(data)
result = compute_did(df)

# treat_pre=11, treat_post=21, control_pre=11, control_post=14
# delta_treat=10, delta_control=3, did=7
assert result['did_estimate'] == 7.0
assert result['conclusion'] == 'Positive effect'
print("All tests passed ✅")
print(result)
for k, v in result.items():
    print(f"{k:>16}: {v}")

All tests passed ✅
{'treat_pre': np.float64(11.0), 'treat_post': np.float64(21.0), 'control_pre': np.float64(11.0), 'control_post': np.float64(14.0), 'delta_treat': np.float64(10.0), 'delta_control': np.float64(3.0), 'did_estimate': np.float64(7.0), 'conclusion': 'Positive effect'}
       treat_pre: 11.0
      treat_post: 21.0
     control_pre: 11.0
    control_post: 14.0
     delta_treat: 10.0
   delta_control: 3.0
    did_estimate: 7.0
      conclusion: Positive effect


In [15]:
import statsmodels.api as sm

def test_parallel_trends(df, group_col='group', time_col='week', outcome_col='outcome'):
    """
    Run on PRE-PERIOD data only.
    H₀: Treat × Time coefficient = 0  (parallel trends holds)
    p < 0.05 → assumption violated → DiD is invalid
    """
    df = df.copy()
    df['treat']       = (df[group_col] == 'treat').astype(int)
    df['interaction'] = df['treat'] * df[time_col]

    X     = sm.add_constant(df[['treat', time_col, 'interaction']])
    model = sm.OLS(df[outcome_col], X).fit()

    coef  = model.params['interaction']
    pval  = model.pvalues['interaction']
    holds = pval >= 0.05

    return {
        'interaction_coef'     : coef,
        'interaction_pval'     : pval,
        'parallel_trends_holds': holds,
        'conclusion': 'Parallel trends holds ✅' if holds else 'Parallel trends VIOLATED ❌'
    }